In [1]:
import os
import grpc
import asyncio
from threading import Thread

Use <i>protoc</i> to generate <i>pb2.py</i> and <i>pb2_grpc.py</i> stub files for protos and services accordingly. Note, you need to have the following packages installed:

<ul>
    <li>protobuf</li>
    <li>grpcio</li>
    <li>grpcio-tools</li>
</ul>


In [2]:
# Use protoc to generate pb2 files from items.proto
# If any of the files need to go to any other folders, provide directories accordingly
os.system("python -m grpc_tools.protoc --proto_path=. --python_out=. --grpc_python_out=. items.proto")

0

In [3]:
# Import freely as they are generated in the current level
import items_pb2
import items_pb2_grpc

# See my_server.py that implements items_pb2_grpc.MessageServiceServicer
# Its serve() method publishes all of its gRPC services
import my_server

<h3>Start the Server</h3>

In [4]:
# Start only for 60 minutes
# Should not block the execution, and hence using another thread
_thread = Thread(target=asyncio.run, args=(my_server.serve(timeout=60),))
_thread.start()

About to start the server...


<h3>gRPC Client -- Create Channel and Stub</h3>

In [5]:
channel = grpc.insecure_channel('localhost:50051')

In [6]:
stub = items_pb2_grpc.MessageServiceStub(channel)

<h3>Call Regular gRPC API</h3>

In [ ]:
response = stub.UnaryGetMessage(items_pb2.MessageRequest(message="message"))
response

message: "message RESPONSE"

<h3>Call gRPC API with Streaming Input</h3>

In [ ]:
def generate_requests(messages):
    """Message stream generator."""
    for message in messages:
        request = items_pb2.MessageRequest(message=message)
        yield request
        
response = stub.ClientStreamGetMessage(generate_requests(["one", "two", "three"]))
response

message: "one,two,three RESPONSE"

<h3>Call Streaming gRPC API</h3>

In [ ]:
request = items_pb2.MessageRequest(message="Hello")
for response in stub.ServerStreamGetMessage(request):
    print(response)

message: "Hello 1 RESPONSE"

message: "Hello 2 RESPONSE"

message: "Hello 3 RESPONSE"

message: "Hello 4 RESPONSE"

message: "Hello 5 RESPONSE"

message: "Hello 6 RESPONSE"

message: "Hello 7 RESPONSE"

message: "Hello 8 RESPONSE"

message: "Hello 9 RESPONSE"



<h3>Call Bi-directional gRPC Streaming API</h3>

In [ ]:
for response in stub.StreamGetMessage(generate_requests(["one", "two", "three", "four"])):
    print(response)

message: "one,two,three,four 1 RESPONSE"

message: "one,two,three,four 2 RESPONSE"

message: "one,two,three,four 3 RESPONSE"

message: "one,two,three,four 4 RESPONSE"

message: "one,two,three,four 5 RESPONSE"

message: "one,two,three,four 6 RESPONSE"

message: "one,two,three,four 7 RESPONSE"

message: "one,two,three,four 8 RESPONSE"

message: "one,two,three,four 9 RESPONSE"



In [ ]:
request = items_pb2.MessageRequest(message='blah')

In [ ]:
num = 3
response = items_pb2.MessageResponse(message = f"{request.message} RESPONSE")